# Aula 14 — LLM Foundations: da classificação à geração

Esta aula inaugura o bloco **LLM Foundations** do TIL.

A pergunta central é:

> **O que muda quando deixamos de apenas classificar texto e passamos a gerar texto token a token?**

A aula é intencionalmente pequena, executável com **Internet OFF** e sem API proprietária.

## Objetivos

Ao final, você deverá conseguir:

- distinguir classificação de geração autoregressiva;
- explicar next-token prediction;
- interpretar logits e probabilidades em nível conceitual;
- comparar greedy decoding e sampling;
- observar o efeito de temperature, top-k e top-p;
- diferenciar texto livre de structured output;
- identificar failure modes básicos;
- relacionar geração a custo, latência, risco e utility;
- justificar quando um LLM adiciona capacidade necessária — e quando não adiciona.

## Glossário da aula

Os conceitos centrais desta aula já estão integrados ao **Glossário Vivo**:

**[LLM](https://github.com/pedroregato/text-intelligence-lab/blob/main/docs/glossary/glossary.pt-BR.md#large-language-model-llm) · [Modelo autoregressivo](https://github.com/pedroregato/text-intelligence-lab/blob/main/docs/glossary/glossary.pt-BR.md#modelo-autoregressivo) · [Prompt](https://github.com/pedroregato/text-intelligence-lab/blob/main/docs/glossary/glossary.pt-BR.md#prompt) · [Contexto](https://github.com/pedroregato/text-intelligence-lab/blob/main/docs/glossary/glossary.pt-BR.md#contexto) · [Janela de contexto](https://github.com/pedroregato/text-intelligence-lab/blob/main/docs/glossary/glossary.pt-BR.md#janela-de-contexto) · [Logits](https://github.com/pedroregato/text-intelligence-lab/blob/main/docs/glossary/glossary.pt-BR.md#logits) · [Sampling](https://github.com/pedroregato/text-intelligence-lab/blob/main/docs/glossary/glossary.pt-BR.md#sampling) · [Temperature](https://github.com/pedroregato/text-intelligence-lab/blob/main/docs/glossary/glossary.pt-BR.md#temperature) · [Top-k](https://github.com/pedroregato/text-intelligence-lab/blob/main/docs/glossary/glossary.pt-BR.md#top-k) · [Top-p](https://github.com/pedroregato/text-intelligence-lab/blob/main/docs/glossary/glossary.pt-BR.md#top-p) · [Greedy decoding](https://github.com/pedroregato/text-intelligence-lab/blob/main/docs/glossary/glossary.pt-BR.md#greedy-decoding) · [Saída estruturada](https://github.com/pedroregato/text-intelligence-lab/blob/main/docs/glossary/glossary.pt-BR.md#saída-estruturada) · [Geração](https://github.com/pedroregato/text-intelligence-lab/blob/main/docs/glossary/glossary.pt-BR.md#geração) · [Alucinação](https://github.com/pedroregato/text-intelligence-lab/blob/main/docs/glossary/glossary.pt-BR.md#alucinação)**

As versões PT-BR, EN e HTML são derivadas da fonte estruturada do glossário.


## 1. Classificar não é gerar

Um classificador recebe uma entrada e escolhe uma classe dentro de um espaço de decisão definido.

```text
entrada
→ representação
→ probabilidades por classe
→ classe
```

Um modelo autoregressivo trabalha de outra forma:

```text
contexto
→ distribuição para o próximo token
→ token escolhido
→ contexto atualizado
→ nova distribuição
→ ...
```

A diferença central não é “modelo simples versus modelo inteligente”. É **tipo de tarefa e capacidade oferecida**.


In [ ]:
import math
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

print("Ambiente da Aula 14 pronto.")
print("Internet não é necessária para os experimentos centrais.")


## 2. Um modelo de próximo token em miniatura

Vamos trabalhar com um vocabulário didático. Os valores abaixo são **logits sintéticos**: não são saída de um LLM real e não devem ser interpretados como benchmark.

O objetivo é observar a transformação:

```text
logits
→ probabilidades
→ política de escolha
→ próximo token
```


In [ ]:
tokens = np.array(["ótimo", "bom", "possível", "ruim", "incerto"])
logits = np.array([2.2, 1.5, 0.7, -0.2, 0.3], dtype=float)

def softmax(x):
    z = x - np.max(x)
    e = np.exp(z)
    return e / e.sum()

base_probs = softmax(logits)
dist = pd.DataFrame({"token": tokens, "logit": logits, "probability": base_probs})
display(dist.sort_values("probability", ascending=False).reset_index(drop=True))

fig, ax = plt.subplots(figsize=(8, 4))
ax.bar(dist["token"], dist["probability"])
ax.set(ylabel="Probabilidade", title="Distribuição didática do próximo token")
ax.grid(axis="y", alpha=.25)
plt.show()
plt.close(fig)


### Observe

- o maior logit produz a maior probabilidade;
- a distribuição mantém alternativas plausíveis;
- **greedy decoding** escolheria sempre o token de maior probabilidade;
- **sampling** permite escolher segundo a distribuição.

A capacidade de gerar não depende apenas da distribuição produzida pelo modelo, mas também da política usada para selecionar o próximo token.


## 3. Temperature: mudar a distribuição, não “a inteligência”

Uma forma simples de aplicar temperature é:

```text
softmax(logits / temperature)
```

Temperature menor tende a concentrar a distribuição. Temperature maior tende a distribuí-la mais.

A interpretação correta é probabilística. Evite reduzir temperature a um botão de “criatividade”.


In [ ]:
def probabilities_with_temperature(logits, temperature):
    if temperature <= 0:
        raise ValueError("temperature deve ser > 0")
    return softmax(logits / temperature)

temperatures = [0.3, 0.7, 1.0, 1.5]
rows = []

for t in temperatures:
    p = probabilities_with_temperature(logits, t)
    for token, prob in zip(tokens, p):
        rows.append({"temperature": t, "token": token, "probability": prob})

temp_df = pd.DataFrame(rows)
display(temp_df.pivot(index="token", columns="temperature", values="probability").round(4))

fig, ax = plt.subplots(figsize=(8, 4))
for t in temperatures:
    p = probabilities_with_temperature(logits, t)
    ax.plot(tokens, p, marker="o", label=f"T={t}")
ax.set(ylabel="Probabilidade", title="Efeito da temperature")
ax.legend()
ax.grid(alpha=.25)
plt.show()
plt.close(fig)


## 4. Top-k e top-p

Duas estratégias comuns limitam quais tokens permanecem candidatos.

- **top-k**: mantém os `k` tokens mais prováveis;
- **top-p**: mantém o menor conjunto cuja probabilidade acumulada atinge um limiar `p`.

Essas técnicas mudam o espaço de escolha. Elas não garantem factualidade, segurança ou qualidade.


In [ ]:
def top_k_distribution(tokens, probs, k):
    idx = np.argsort(probs)[::-1][:k]
    kept = np.zeros_like(probs)
    kept[idx] = probs[idx]
    kept = kept / kept.sum()
    return pd.DataFrame({"token": tokens, "probability": kept})

def top_p_distribution(tokens, probs, p):
    order = np.argsort(probs)[::-1]
    sorted_probs = probs[order]
    cumulative = np.cumsum(sorted_probs)
    cutoff = np.searchsorted(cumulative, p, side="left") + 1
    keep_idx = order[:cutoff]
    kept = np.zeros_like(probs)
    kept[keep_idx] = probs[keep_idx]
    kept = kept / kept.sum()
    return pd.DataFrame({"token": tokens, "probability": kept})

display(top_k_distribution(tokens, base_probs, k=2).round(4))
display(top_p_distribution(tokens, base_probs, p=0.80).round(4))


## 5. Texto livre versus structured output

Um modelo pode produzir texto linguisticamente plausível e, ainda assim, falhar em um contrato de dados.

Imagine que um sistema espera:

```json
{
  "sentiment": "positive|neutral|negative",
  "confidence": 0.0
}
```

Gerar e **validar** são etapas diferentes.


In [ ]:
def validate_sentiment_output(obj):
    allowed = {"positive", "neutral", "negative"}

    if not isinstance(obj, dict):
        return False, "A saída deve ser um objeto/dict."

    if set(obj) != {"sentiment", "confidence"}:
        return False, "Campos esperados: sentiment e confidence."

    if obj["sentiment"] not in allowed:
        return False, "sentiment fora do conjunto permitido."

    if not isinstance(obj["confidence"], (int, float)):
        return False, "confidence deve ser numérica."

    if not 0 <= obj["confidence"] <= 1:
        return False, "confidence deve estar entre 0 e 1."

    return True, "válido"

examples = [
    {"sentiment": "positive", "confidence": 0.91},
    {"sentiment": "positivo", "confidence": 0.91},
    {"sentiment": "neutral", "confidence": 1.4},
    {"sentiment": "negative", "confidence": "alta"},
]

validation = []
for x in examples:
    ok, reason = validate_sentiment_output(x)
    validation.append({"output": x, "valid": ok, "reason": reason})

display(pd.DataFrame(validation))


## 6. Failure modes: geração plausível não é garantia

Modelos generativos podem apresentar diferentes tipos de falha:

- conteúdo plausível porém incorreto;
- perda de restrições;
- formato inválido;
- sensibilidade a pequenas mudanças de contexto;
- variabilidade entre execuções;
- resposta excessivamente longa;
- custo e latência maiores que o necessário.

É útil tratar esses fenômenos separadamente. Colocar todo erro sob o rótulo “hallucination” pode esconder causas diferentes e, portanto, estratégias de mitigação diferentes.


## 7. Utility da geração

A Aula 13C introduziu a ideia:

```text
utility = f(qualidade, custo, latência, risco, autonomia)
```

Na Aula 14, a principal capacidade nova é **geração**. Autonomia ainda é baixa: não estamos ensinando agentes.

Pergunta de engenharia:

> A capacidade generativa resolve uma necessidade real desta tarefa que um classificador, uma regra ou um encoder não resolve adequadamente?


## 8. Exercício 1 — Greedy vs sampling

Considere a distribuição `base_probs`.

**Sua tarefa:** identifique o token escolhido por greedy decoding e liste os dois tokens disponíveis sob `top-k = 2`.

Escreva e execute sua solução na célula abaixo.


In [ ]:
# Sua resposta aqui


### Dica

Use `np.argmax()` para localizar o maior valor e `np.argsort()` para ordenar índices.


In [ ]:
# Solução executável
greedy_token = tokens[np.argmax(base_probs)]
top2_idx = np.argsort(base_probs)[::-1][:2]
top2_tokens = tokens[top2_idx].tolist()

print("Greedy:", greedy_token)
print("Top-k=2:", top2_tokens)


## 9. Exercício 2 — Validando structured output

Implemente uma chamada de teste para `validate_sentiment_output()` usando um objeto criado por você.

Depois, altere um campo para tornar a saída inválida e observe a mensagem de validação.


In [ ]:
# Sua resposta aqui


### Dica

Crie primeiro um `dict` com exatamente os campos `sentiment` e `confidence`. Depois mude um valor para violar uma das regras.


In [ ]:
# Solução executável
valid_example = {"sentiment": "positive", "confidence": 0.88}
invalid_example = {"sentiment": "positive", "confidence": 1.20}

print(validate_sentiment_output(valid_example))
print(validate_sentiment_output(invalid_example))


## 10. Exercício 3 — Escolha a arquitetura mais simples que atende à tarefa

Para cada situação abaixo, escolha inicialmente entre:

- regra;
- classificador;
- encoder/Transformer;
- LLM generativo.

Situações:

1. detectar se uma mensagem pertence a uma de três categorias fixas;
2. produzir um resumo textual livre de um atendimento longo;
3. verificar se uma string segue um formato determinístico simples;
4. calcular embeddings semânticos para busca por similaridade.

Não existe um “vencedor universal”. Justifique a **capacidade necessária** e o custo de adicionar complexidade.


### Resposta de referência

Uma resposta coerente pode ser:

1. **classificador** — espaço de classes fechado; geração não é requisito;
2. **LLM generativo** — a saída é texto novo e variável;
3. **regra** — se o formato for determinístico e simples, uma expressão regular ou parser pode bastar;
4. **encoder/Transformer** — a tarefa pede representação semântica, não necessariamente geração.

O ponto do exercício é aplicar a regra do TIL:

```text
capacidade adicional
→ evidência adicional
→ complexidade justificada
```


## 11. Reprodutibilidade

Esta versão da aula:

- não treina LLM;
- não chama API externa;
- não exige Internet;
- usa distribuições sintéticas explicitamente didáticas;
- usa apenas NumPy, pandas e Matplotlib;
- não apresenta os resultados como benchmark real.

Quando medições reais de modelos generativos forem incorporadas, elas deverão vir de um experimento separado na trilha **AUTHOR / EVIDENCE**.


## 12. Síntese

Você deve sair desta aula distinguindo três coisas:

```text
modelo que classifica
≠
modelo que representa
≠
modelo que gera
```

A capacidade generativa amplia o espaço de problemas que podemos resolver, mas traz também novos custos, latência, variabilidade e failure modes.

### Próxima ponte

Agora podemos fazer uma pergunta nova:

> Se o modelo gera bem, mas não possui no contexto a informação necessária, como fornecer evidência externa de forma controlada?

Essa pergunta abre o próximo bloco: **Retrieval and Grounding**.
